# 36 · 规范化、中文切分与新词

单元：S03。先修：35。

来源：[王树森固定目录](https://github.com/wangshusen/SearchEngine/tree/7339c6b126e314abdb74a89adb033d39d8e3d37b)；[Manning等《Introduction to Information Retrieval》](https://nlp.stanford.edu/IR-book/)（补充原始教材；章节1、2、6、8）。这是 Lab 自编实验；王树森后四模块只有公开目录，不称原作者完整讲义。

证据边界：人工中文词例；字典机制，非神经中文模型训练。

学习方式：先预测每个对照，再只改一个变量，运行后解释差异。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from reclab.learning.search_curriculum import *

## 规范化契约
NFKC统一全半角，lower合并英文大小写；可能损失区分信息。预测规范化是否幂等。

In [2]:
x='ＡＩ  手机  Battery'
print(x,'→',normalize(x))
assert normalize(normalize(x))==normalize(x)

ＡＩ  手机  Battery → ai 手机 battery


## 双向最大匹配与歧义
左到右取最长词不保证全局正确。预测研究生命起源的两种切分；固定词典，只改变方向。

In [3]:
vocab={'研究','研究生','生命','命','起源'}
f=segment('研究生命起源',vocab); b=segment('研究生命起源',vocab,reverse=True)
print('forward',f,'backward',b)
assert f!=b and ''.join(f)==''.join(b)

forward ['研究生', '命', '起源'] backward ['研究', '生命', '起源']


## OOV与新词候选
新词发现需频次、凝固度和上下文自由度等证据。这里仅用人工重复短语的bigram计数提出候选，不能证明它是词。只添加一个词。

In [4]:
samples=['量子芯片发布','量子芯片测试','量子芯片量产']
counts=Counter(s[i:i+2] for s in samples for i in range(len(s)-1))
print(counts.most_common(5))
print(segment('量子芯片',{'芯片'}),segment('量子芯片',{'芯片','量子'}))
assert segment('量子芯片',{'芯片','量子'})==['量子','芯片']

[('量子', 3), ('子芯', 3), ('芯片', 3), ('片发', 1), ('发布', 1)]
['量', '子', '芯片'] ['量子', '芯片']


## 神经子词不是中文分词
读取已有英文TinyBERT tokenizer，不下载新模型。WordPiece以模型词表切分并可能产生UNK；它不是监督中文词边界模型，也不能用这个英文模型证明中文切词质量。

In [5]:
from reclab.search_lessons import models
tok=models().pair_tokenizer
for s in ['研究生命起源','electroencephalography']:
    print(s,tok.tokenize(s))
print('vocab',tok.vocab_size)

/Users/lynnlee/Documents/Codex/2026-10-30/task/soutui-lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


研究生命起源 ['[UNK]', '[UNK]', '生', '[UNK]', '[UNK]', '[UNK]']
electroencephalography ['electro', '##ence', '##pha', '##log', '##raphy']
vocab 30522


## 本次保存结果的解读

正向最长匹配得到“研究生/命/起源”，反向得到“研究/生命/起源”；两者都能还原字符串，但词义边界不同。添加“量子”只改变OOV切分，重复bigram本身不能证明新词。英文TinyBERT对中文产生多个UNK，不能拿其子词输出评估中文分词。

改参数后应以新输出重新判断，不把这段固定解读当作新的测量。

## 复盘与练习

写下输入、输出、保持不变的量与本章不能支持的结论。选一个上面的单变量参数先预测再重跑；保留失败例。回到搜索学习路径继续下一语义单元。